# Example of cart pole simulation in Mujoco

This tutorial introduces how to set up closed-loop control system for a simple cart pole systems in Mujoco.

First, let's create the cart pole model

In [475]:
import mujoco
import numpy as np

import mujoco.viewer as viewer
modelxml = """
<mujoco model="CartPole">
  <compiler eulerseq="XYZ"/>
  <visual>
  <headlight ambient="0.5 0.5 0.5"
             diffuse="1 1 1"/>
  </visual>
  <default>
    <default class="unused"/>
  </default>
  <asset>
    <texture name="grid" type="2d" builtin="checker" rgb1="0.1 0.2 0.3" rgb2="0.2 0.3 0.4" width="512" height="512"/>
    <material name="grid" class="unused" texture="grid" texrepeat="1 1" texuniform="true" reflectance="0.2"/>
  </asset>
  <worldbody>
    <geom name="floor" class="unused" type="plane" condim="3" size="0 0 0.05" material="grid" pos="0 0 -1"/>
    <body name="Cart" pos="0 0 0" euler="0 -0 0">

    <!-- change inertia, different from sdf -->
    <!-- For this model case, with the cart not having any rotational
             degrees of freedom, the values of the inertia matrix do not
             participate in the model. Therefore we just set them to zero
             (or near to zero since sdformat does not allow exact zeroes
             for inertia values). -->

      <inertial pos="0 0 0" mass="5" diaginertia="0.00000000001 0.00000000001 0.00000000001"/>
      <geom name="cart_visual" class="unused" type="box" contype="0" conaffinity="0" group="0" size="0.12 0.06 0.06" pos="0 0 0" euler="0 -0 0"/>
      <joint name="CartSlider" class="unused" type="slide" pos="0 0 0" axis="1 0 0" limited="true" range="-2 2"/>
      <body name="Pole" pos="0 0 -0.5" euler="0 -0 0">
        <inertial pos="0 0 0" mass="1" diaginertia="0.00000000001 0.00000000001 0.00000000001"/>
        <geom name="pole_point_mass" class="unused" type="sphere" contype="0" conaffinity="0" group="0" size="0.05" pos="0 0 0" euler="0 -0 0"/>
        <geom name="pole_rod" class="unused" type="cylinder" contype="0" conaffinity="0" group="0" size="0.025 0.25" pos="0 0 0.25" euler="0 -0 0"/>
        <joint name="PolePin" class="unused" type="hinge" pos="0 0 0.5" axis="0 -1 0"/>
      </body>
    </body>
  </worldbody>
  <sensor>
        <!-- joint position sensing -->
        <jointpos joint="CartSlider" name="cart_p"/>
        <jointpos joint="PolePin" name="pole_theta"/>
        
        <!-- joint velocity sensing -->
        <jointvel joint="CartSlider" name="cart_v"/>
        <jointvel joint="PolePin" name="pole_w"/>
  </sensor>
  <actuator>
    <motor joint="CartSlider"/>
  </actuator>
  <keyframe>
    <key name="off1" qpos="0 3"/>
  </keyframe>
</mujoco>
"""
model = mujoco.MjModel.from_xml_string(modelxml)
data = mujoco.MjData(model)

# 初始化状态到off1
init_state_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_KEY, "off1") # 0
mujoco.mj_resetDataKeyframe(model, data, init_state_id)  # Reset the state to keyframe 0

# 可视化
# mujoco.viewer.launch(model, data)

In [476]:
print([model.joint(i).name for i in range(model.njnt)]) # 打印所有关节
print([model.sensor(i).name for i in range(model.nsensor)]) # 打印所有传感器
data.qvel
data.qpos

['CartSlider', 'PolePin']
['cart_p', 'pole_theta', 'cart_v', 'pole_w']


array([0., 3.])

In [477]:
def PID(data):
    x, theta = data.qpos
    x_dot, theta_dot = data.qvel

    # 以竖直向上的 π 为目标，角度误差折回 [-π, π)
    error = theta % (2 * np.pi) - np.pi

    force = (
        14.1 * x
        + 19.1 * x_dot
        - 184.8 * error
        - 39.4 * theta_dot
    )
    return float(np.clip(force, -50.0, 50.0))

In [478]:
def PID_bad(data):
    """
    与效果好的pid做对比
    """
    x, theta = data.qpos
    x_dot, theta_dot = data.qvel

    # 以竖直向上的 π 为目标，角度误差折回 [-π, π)
    error = theta % (2 * np.pi) - np.pi

    force = (
        10 * x +
        -1000 * error
        - 30 * theta_dot
    )
    return float(np.clip(force, -50.0, 50.0))

In [479]:
import mujoco.viewer as viewer
import time

mujoco.mj_resetDataKeyframe(model, data, init_state_id)  # Reset the state to keyframe 0

with viewer.launch_passive(model, data) as viewer:
  while viewer.is_running():
    start = time.perf_counter()
    data.ctrl = PID_bad(data)
    mujoco.mj_step(model, data)

    viewer.sync()
    time_until_next_step = max(model.opt.timestep - (time.perf_counter() - start), 0.0)
    time.sleep(time_until_next_step)